# Critique, revision and acceptance are three different mechanisms

Day 26 · Chapter 15 · bounded CPU extension.

A critique says “change the answer.” The revision is well formed and the acceptance gate approves it. Has the system improved? What if the next critique reverses that change?

This notebook studies an original state machine, twelve deliberately authored failure cases, exact serialized-token programmatic controls and replay of 864 actual DXI05 tiny-model candidates. Critiques and revisions are **programmatic, not neural self-critiques**. The historical tiny models do not see the critiques, and no model is invoked here. References grade decisions afterward; they never steer callbacks or stopping.

Read the [frozen specification](../../experiments/specs/2026-10-04-critique-revision.md), [original fixtures](../../fixtures/critique-revision/README.md) and [measured report](../../experiments/reports/2026-10-04-critique-revision.md) alongside this lesson. Source notebooks stay unexecuted; fresh copies supply the previews. Running code regenerates plots from the frozen measured rows or current live state.

Independent review found a stopping-contract gap in the first instrument: an EOS-looking critique marked token-capped could trigger revision. The [hardening follow-up](../../experiments/specs/2026-10-04-critique-revision-hardening-final.md) requires natural critique EOS and retains capped output while skipping revision. This notebook loads the final current-source campaign; original and intermediate snapshots remain historical, not silently relabeled.

In [ ]:
from pathlib import Path
import sys, json
from copy import deepcopy
from collections import Counter
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src/dongxi_llms").is_dir())
if str(root/"src") not in sys.path: sys.path.insert(0,str(root/"src"))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import ListedColormap
from dongxi_llms.critique_revision_lab import (ACTION_IDS, EOS, NUMBER_START, callback_views,
    digest, emission, independent_control, load_inputs, programmatic_pool, run_campaign,
    run_loop, stable_payload)
directory=root/"experiments/reports/2026-10-04-critique-revision-hardening-final"
assert digest(directory/"results.json")=="41bf5995ff114358f03baa7a84f17496086993bd7098d4e206a148e0790f6b4f"
measured=json.loads((directory/"results.json").read_text())
for path,expected in measured["source_identity"]["source_and_input_sha256"].items():
    assert digest(root/path)==expected,path
protocol,items,source_rows,source_pools,fixtures,audit=load_inputs(root,root/"fixtures/critique-revision/protocol.json")
plt.rcParams.update({"figure.figsize":(10,4.2),"font.size":10,"font.family":"DejaVu Sans",
                     "axes.spines.top":False,"axes.spines.right":False})
colors={"baseline":"#777777","revision":"#cb7844","independent":"#35779e",
        "right":"#5a8a68","wrong":"#bf6657","invalid":"#929292"}
all_rows=[r for r in measured["summary"] if r["slice"]=="all"]
print("Frozen panels:",len(measured["cases"]),"case/mode rows;",len(source_rows),"historical model candidates.")
print("New model forwards:",measured["new_model_forwards"],"— callbacks are programmatic.")


### Prediction

## 1. Draw the control boundary before inspecting accuracy

Exercise: where would a hidden reference leak make the system look artificially good? Does accepting a tie mean the answer remains the same?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
fig,ax=plt.subplots(figsize=(12,4.2))
ax.set(xlim=(0,13),ylim=(0,4));ax.axis("off")
boxes=[(.15,2.0,2.2,"Current draft\nraw IDs + stop/error"),
       (3.2,2.0,2.3,"Programmatic critique\nknown action\nnatural EOS"),
       (6.35,2.0,2.25,"Programmatic revision\nproposal, not verdict"),
       (9.55,2.0,2.75,"Acceptance gate\nsingle binary + EOS")]
for x,y,w,text in boxes:
    ax.add_patch(FancyBboxPatch((x,y),w,.95,boxstyle="round,pad=.08",facecolor="#f5f5f5",edgecolor="#777"))
    ax.text(x+w/2,y+.475,text,ha="center",va="center")
for start,end in ((2.4,3.08),(5.58,6.23),(8.68,9.43)):
    ax.annotate("",xy=(end,2.47),xytext=(start,2.47),arrowprops={"arrowstyle":"->","color":"#555"})
ax.plot([10.8,10.8,1.25],[1.92,1.55,1.55],color="#cb7844",linewidth=1.2)
ax.annotate("",xy=(1.25,1.92),xytext=(1.25,1.55),arrowprops={"arrowstyle":"->","color":"#cb7844"})
ax.text(6.5,1.2,"accepted state feeds next round",ha="center",color="#9c542d")
ax.text(6.5,.45,"Evaluation-only reference → attempted and delivered grades\nNo grade/reference in callback inputs or stopping decisions",
        ha="center",va="center",bbox={"facecolor":"#edf3f7","edgecolor":"#35779e","boxstyle":"round,pad=.3"})
ax.set_title("Original replay control structure — no neural critic or revision model",pad=12)
plt.show()


### Reference explanation

Reference: the critique sees only a whitelisted question and current draft. The revision sees that draft plus a symbolic critique. The acceptance gate checks one binary token followed by EOS; it accepts different well-formed values as a tie. Only the evaluator sees the reference. Stop after an identical **accepted** token path or two rounds, never after a correct grade.

The schematic below is structure, not measured model latency.

![Critique and revision control boundary](../figures/chapter-15/day-26-04_critique_revision_and_acceptance-01.png)


### Prediction

## 2. A correct draft can be revised into a wrong answer

Predict the one-round and two-round endpoints when the critic always asks FLIP. Then change `max_rounds=1` below and rerun. This is a mechanism intervention, not a new campaign or a selected favorable result.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
example=deepcopy(items[0])
start=emission([NUMBER_START+int(example["reference"]),EOS])
live=run_loop(example,start,mode="contrarian",max_rounds=2)
print("Live accepted transitions:",[r["delivered_transition"]["category"] for r in live["rounds"]])
print("Finally delivered:",live["delivered"]["raw_response"],live["delivered_grade"]["state"])
states={"invalid":0,"wrong":1,"right":2}
names=[r["fixture"]["id"] for r in measured["adversarial"]]
paths=[]
for row in measured["adversarial"]:
    loop=row["loop"]
    values=[loop["initial_grade"]["state"]]+[r["delivered_transition"]["after"] for r in loop["rounds"]]
    paths.append(values+[values[-1]]*(3-len(values)))
arr=np.array([[states[s] for s in p] for p in paths])
fig,ax=plt.subplots(figsize=(10,6))
ax.imshow(arr,cmap=ListedColormap([colors["invalid"],colors["wrong"],colors["right"]]),vmin=-.5,vmax=2.5,aspect="auto")
for y,path in enumerate(paths):
    for x,s in enumerate(path):ax.text(x,y,s,ha="center",va="center",color="white",fontweight="bold")
ax.set(xticks=range(3),xticklabels=["Initial draft","Delivered round 1","Delivered round 2"],
       yticks=range(len(names)),yticklabels=names,title="Authored adversarial fixtures — chosen tests, not model accuracy")
ax.set_xticks(np.arange(-.5,3,1),minor=True);ax.set_yticks(np.arange(-.5,len(names),1),minor=True)
ax.grid(which="minor",color="white",linewidth=2);ax.tick_params(which="minor",bottom=False,left=False)
fig.tight_layout();plt.show()


### Reference explanation

Reference: a format gate accepts both directions. One flip changes a correct binary answer into a wrong one; a second flip restores it. Reporting only the final answer hides the harmful intermediate round. Reporting only the best round hides the final delivered failure. The authored tests below deliberately include both patterns and retain invalid and exception paths.

![Every authored round, including invalid drafts](../figures/chapter-15/day-26-04_critique_revision_and_acceptance-02.png)


### Prediction

## 3. Attempted proposals and accepted states must both survive

Why isn't “we rejected the bad proposal” a reason to discard its cost or raw output?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
panels=["exact-programmatic-control","actual-candidate-replay"]
labels=["Programmatic exact-token panel","Historical actual-candidate replay"]
categories=["wrong_to_right","right_to_wrong","invalid","unchanged"]
rows=[next(r for r in all_rows if r["panel"]==p and r["mode"]=="contrarian") for p in panels]
fig,ax=plt.subplots(figsize=(10,4.5))
x=np.arange(len(categories));width=.34
for j,(row,label,color) in enumerate(zip(rows,labels,["#35779e","#cb7844"])):
    counts=[row["attempted_transitions"].get(k,0) for k in categories]
    bars=ax.bar(x+(j-.5)*width,counts,width,label=label,color=color)
    ax.bar_label(bars,padding=3)
ax.set(xticks=x,xticklabels=["Wrong → right","Right → wrong","Changed / invalid","Unchanged path"],
       ylabel="Attempted round proposals (count)",title="Round-level accounting, not only favorable revisions")
ax.legend(frameon=False);fig.tight_layout();plt.show()
for panel in panels:
    chosen=[c for c in measured["cases"] if c["panel"]==panel]
    invalid=sum(r["attempted_transition"]["after"]=="invalid" for c in chosen for r in c["loop"]["rounds"])
    print(panel,"invalid attempted proposals across all modes:",invalid)
errors=[(c["fixture"]["id"],r["round"],k["role"],k["error"]) for c in measured["adversarial"]
        for r in c["loop"]["rounds"] for k in r["callback_costs"] if k["error"]]
print("Retained authored exceptions/validation errors:",errors)
capped=run_loop(example,start,critique_fn=lambda *args: emission([ACTION_IDS["keep"],EOS],"max_tokens"))
assert all(r["revision"] is None and not r["acceptance"]["accepted"] for r in capped["rounds"])
assert capped["serialized_tokens"]==6
print("Capped critique retained, charged and cannot trigger revision:",capped["rounds"][0]["critique"])


### Reference explanation

Reference: a rejected proposal still happened. It consumes emitted symbols or callback work, and its failure identifies what the gate prevented. A critique exception skips revision; a revision exception preserves the current draft. Neither is a successful unchanged revision.

The plot counts attempted proposals in the fixed contrarian panels. The separate invalid-after count printed below also includes unchanged invalid paths, which cannot be inferred from a transition label alone. Source siblings and repeated mode rows are related, not independent trials.

![Beneficial and harmful attempted changes](../figures/chapter-15/day-26-04_critique_revision_and_acceptance-03.png)


### Prediction

## 4. Compare what is finally delivered

Exercise: does a higher replay success count demonstrate a better reasoning model? What did REPAIR actually learn?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
modes=["identity","repair","contrarian"]
fig,axes=plt.subplots(1,2,figsize=(12,4.6),sharey=True)
for ax,panel,title in zip(axes,panels,labels):
    selected=[next(r for r in all_rows if r["panel"]==panel and r["mode"]==m) for m in modes]
    x=np.arange(3);width=.25
    for offset,key,label,color in [(-1,"no_critique_success","No critique",colors["baseline"]),
                                   (0,"revision_success","Delivered revision",colors["revision"]),
                                   (1,"independent_success","Independent majority",colors["independent"])]:
        vals=[100*r[key]/r["cases"] for r in selected]
        bars=ax.bar(x+offset*width,vals,width,color=color,label=label)
        ax.bar_label(bars,labels=[f'{r[key]}/{r["cases"]}' for r in selected],padding=3,fontsize=8)
    ax.set(xticks=x,xticklabels=modes,ylim=(0,100),title=title)
    ax.set_xlabel("Frozen programmatic critique rule")
axes[0].set_ylabel("Finally delivered binary/EOS success (%)")
axes[1].legend(frameon=False,fontsize=8,loc="upper right")
fig.suptitle("References evaluate afterward — neither critique nor acceptance uses gold",fontsize=11)
fig.tight_layout();plt.show()


### Reference explanation

Reference: it learned nothing. REPAIR keeps the first observed binary token, or defaults to zero, and supplies EOS. It can recover a task answer from a previously invalid path or create a wrong answer that now obeys the format. The model candidates were already collected and are not conditioned on critiques. The two panels cannot be merged into one claimed accuracy.

No-critique spends fewer tokens. Independent majority uses the same frozen candidate pool and gold-blind tie breaking. The real-pool budget is a ceiling with whole-attempt overrun, not exact compute matching.

![Final delivered success in separate panels](../figures/chapter-15/day-26-04_critique_revision_and_acceptance-04.png)


### Prediction

## 5. “Matched budget” needs a unit and a stopping policy

For the exact programmatic panel, the draft costs two symbols, each critique two, and each valid revision two. One unchanged round costs six symbols; two flip rounds cost ten. The independent arm uses three or five two-symbol attempts respectively. EOS is counted.

\[
B_{\mathrm{revision}}=n_{\mathrm{draft}}+\sum_{r=1}^{R}
\left(n_{\mathrm{critique},r}+n_{\mathrm{revision},r}\right).
\]

That equality describes serialized symbolic outputs. It does **not** equate model tokens, FLOPs, elapsed time or API billing. In real-candidate replay, the next whole attempt may overrun the remaining ceiling. We charge it, exclude it from selection and retain its raw output. The plot therefore shows both eligible and attempted tokens. No historical model call is executed or “saved” by the replay.

![Exact serialization matching versus real-pool overshoot](../figures/chapter-15/day-26-04_critique_revision_and_acceptance-05.png)

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
fig,(left,right)=plt.subplots(1,2,figsize=(12,4.5))
program=[next(r for r in all_rows if r["panel"]==panels[0] and r["mode"]==m) for m in modes]
x=np.arange(3);w=.34
a=left.bar(x-w/2,[r["serialized_revision_tokens"] for r in program],w,label="Draft + critique + revision",color=colors["revision"])
b=left.bar(x+w/2,[r["independent_attempted_tokens"] for r in program],w,label="Independent emitted outputs",color=colors["independent"])
left.bar_label(a,padding=3);left.bar_label(b,padding=3)
left.set(xticks=x,xticklabels=modes,ylabel="Serialized output symbols, EOS included",
         title="Exact programmatic matching: 162/162 cases")
left.legend(frameon=False,fontsize=8)
actual=[c for c in measured["cases"] if c["panel"]==panels[1]]
ceiling=np.array([c["independent"]["token_ceiling"] for c in actual])
eligible=np.array([c["independent"]["eligible_tokens"] for c in actual])
attempted=np.array([c["independent"]["attempted_tokens"] for c in actual])
right.scatter(ceiling,eligible,s=36,alpha=.22,color=colors["right"],label="Eligible prefix outputs")
right.scatter(ceiling,attempted,s=22,alpha=.22,color=colors["wrong"],marker="x",label="Attempted outputs, overrun charged")
limits=[0,max(ceiling.max(),attempted.max())+1]
right.plot(limits,limits,"--",color="#666",linewidth=1,label="Exact serialized equality")
right.set(xlim=limits,ylim=limits,xlabel="Revision arm serialized-output ceiling",
          ylabel="Independent historical outputs (tokens)",title="Actual pool: ceiling ≠ exact spent budget")
right.legend(frameon=False,fontsize=8);fig.tight_layout();plt.show()
print("Actual replay exact matches:",sum(c["independent"]["exact_serialized_match"] for c in actual),"/",len(actual))
print("Underfill:",sum(c["independent"]["underfill"] for c in actual),
      "charged overshoot:",sum(c["independent"]["overshoot"] for c in actual))
print("Unique historical source generation tokens:",sum(r["cost"]["generation_tokens"] for r in source_rows))
print("New critique/revision model-forward calls: 0. Reusing pools is not repeated model generation.")


### Reference explanation

Exact programmatic matching counts serialized symbols, including EOS. The actual replay plots eligible cost separately from all attempted cost, charging any overshoot while excluding its response from selection. Reusing historical responses performs zero new critique/revision model forwards and does not establish equal FLOPs, latency or API cost.


### Prediction

## 6. A mean can hide different source and task failures

How could a format repair improve the pooled count while remaining unhelpful on a task family?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
slices=["train","heldout-source","heldout-template","heldout-family"]
fig,axes=plt.subplots(1,2,figsize=(12,5),sharey=True)
for ax,panel,title in zip(axes,panels,labels):
    matrix=[]
    for sl in slices:
        values=[]
        for mode in modes:
            row=next(r for r in measured["summary"] if r["panel"]==panel and r["mode"]==mode and r["slice"]==sl)
            values.extend([row["revision_success"]-row["no_critique_success"],
                           row["independent_success"]-row["no_critique_success"]])
        matrix.append(values)
    a=np.array(matrix)
    ax.imshow(a,cmap="RdBu",vmin=-18,vmax=18,aspect="auto")
    for y in range(a.shape[0]):
        for x in range(a.shape[1]):ax.text(x,y,f"{a[y,x]:+d}",ha="center",va="center",color="black")
    ax.set(yticks=range(4),yticklabels=slices,xticks=range(6),
           xticklabels=["ID\nrev","ID\nind","repair\nrev","repair\nind","flip\nrev","flip\nind"],
           title=title)
fig.suptitle("Success-count difference from no-critique, same frozen item/checkpoint panels",fontsize=11)
fig.tight_layout();plt.show()
for record in measured["adversarial"]:
    rejected=[r["round"] for r in record["loop"]["rounds"] if not r["acceptance"]["accepted"]]
    if rejected:print(record["fixture"]["id"],"rejected rounds:",rejected,
                      "final:",record["loop"]["delivered"]["raw_response"])


### Reference explanation

Reference: the baseline validity, answer balance, template and task family can all differ. The fixed item set contains training, held-out sources, held-out templates and a held-out family. None is selected after observing revision outcomes. Rows below are differences in delivered successes, not statistically independent generalization estimates.

![Fixed slice changes, including negative independent-attempt results](../figures/chapter-15/day-26-04_critique_revision_and_acceptance-06.png)


### Prediction

## 7. Replay the complete frozen comparison

This cell reruns the bounded programmatic state machines and independent selectors, not the historical neural generation. All 864 original responses and their original cost fields must remain byte-identical at the source. The content comparison excludes newly measured callback/campaign wall time and volatile run metadata, not answers, IDs, acceptance, failures, budgets or counts.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
fresh=run_campaign(root,root/"fixtures/critique-revision/protocol.json")
saved={k:v for k,v in measured.items() if k not in ("source_identity","inputs_unchanged","content_sha256")}
assert stable_payload(fresh)==stable_payload(saved)
assert fresh["source_candidates"]==source_rows
assert all(c["independent"]["exact_serialized_match"] for c in fresh["cases"] if c["panel"]=="exact-programmatic-control")
print("All 486 fixed case/mode rows, 12 adversarial paths and 864 source responses replayed.")
print("No changed decisions or emitted IDs; newly measured CPU replay seconds:",round(fresh["campaign_wall_seconds"],3))


### Reference explanation

Reference: a passing replay validates the implemented fixed comparison; it does not validate neural self-critique, a population effect, hidden reasoning faithfulness or production safety. Optional same-model critique generation still needs a separate frozen checkpoint/interface, prompt, budget and actual output protocol.

## Synthesis

A critique produces advice; a revision produces a proposal; an acceptance rule decides what becomes state. None of those roles is automatically a correctness verifier. Inspect proposed changes and delivered states, separate output validity from correctness, and state precisely which budget unit was matched.

The coherent Chapter 15 discussion and worked answers use this notebook as a mechanism companion. Animations, pretrained same-model critique runs and model-scale comparisons are not executed by this lesson.
